# Lesson 4: Tesla's text → numbers (the tokenizer)

**Why:** a model can only do math on numbers, not letters. A **tokenizer** is a two-way dictionary: every character gets a number (its **token id**), and we can translate back and forth.

```
"Tesla"  ──encode──►  [ids]  ──►  the GPT  ──►  [ids]  ──decode──►  "new text"
```

Ours is the simplest possible tokenizer: **one character = one token**. (Lesson 14 builds the smarter kind that GPT-4 uses.)

## 1. The vocabulary

The **vocabulary** is every different character in the text, in sorted order. Its size **V** is how many scores the model outputs at each step (Lesson 3).

In [ ]:
from pathlib import Path
import torch
import matplotlib.pyplot as plt

text = Path("../data/input.txt").read_text(encoding="utf-8")

chars = sorted(set(text))
V = len(chars)
print(f"text length: {len(text):,} characters")
print(f"vocabulary size V = {V}")
print(repr("".join(chars)))

## 2. Encode and decode

Two lookup tables:
- `stoi` ("string to integer"): character → number
- `itos` ("integer to string"): number → character

(`{ch: i for ...}` builds a dictionary in one line: "for each character and its position, map the character to the position".)

**Predict:** in `encode("Tesla")`, will `T` get a **smaller** or **bigger** number than `e`? (Hint: look at the sorted vocabulary above.)

In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}   # 'a' -> number
itos = {i: ch for i, ch in enumerate(chars)}   # number -> 'a'

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

ids = encode("Tesla")
print("encode('Tesla') =", ids)
print("decode(...)     =", decode(ids))

## 3. The whole book as one long tensor

We encode all ~977,000 characters into one long 1-D tensor. `dtype=torch.long` means whole numbers, which is what lookup tables need.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)

print("shape:", data.shape, " dtype:", data.dtype)
print("first 40 ids:", data[:40].tolist())
print("as text:     ", repr(decode(data[:40].tolist())))

## 4. Which characters does Tesla use most?

A first look at what the model will learn: some characters are far more common than others. A model that only learned these frequencies would already beat the know-nothing loss of 4.44.

In [ ]:
counts = torch.bincount(data, minlength=V)
top = counts.argsort(descending=True)[:15]

labels = [repr(itos[i.item()])[1:-1] or "?" for i in top]
labels = ["space" if l == " " else l for l in labels]
plt.figure(figsize=(9, 3))
plt.bar(labels, (counts[top] / len(data) * 100).tolist())
plt.ylabel("% of all characters"); plt.title("15 most common characters in Tesla's text")
plt.show()

## 5. Round trip, and one limitation

A tokenizer must be **lossless**: encoding then decoding the whole book must give back exactly the same text.

The limitation: a character tokenizer only knows characters it has **seen**. Watch what happens with a Serbian letter.

In [ ]:
print("whole book survives the round trip:", decode(encode(text)) == text)

try:
    encode("Nikola Tesla, Smiljan, ć")
except KeyError as err:
    print("can't encode character:", err)

## 6. Final check

In [ ]:
assert decode(encode(text)) == text
assert data.shape == (len(text),) and data.dtype == torch.long
assert data.max().item() == V - 1
print(f"Tokenizer works: {V} characters <-> numbers 0..{V - 1}. Lesson 4 complete.")

## Wrap-up question

Our model can only ever **write** characters that are in its vocabulary. Could it ever write the Serbian letters **ć** or **đ**? Why or why not? What would we need to change?